<a href="https://colab.research.google.com/github/KCDAS35/WEB/blob/main/demo/omnivoice_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice-TTS Colab — T4 Quickstart

**Features:** Zero-shot voice cloning · Cross-lingual TTS (600+ languages) · Voice Design · Batch inference

> **Requirements:** T4 GPU runtime · Model: `k2-fsa/OmniVoice` (auto-downloaded on first run)

## Step 1: GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu} ({vram:.1f} GB VRAM)')
    if vram < 4:
        print('⚠️  WARNING: Less than 4 GB VRAM. Short outputs only — longer text may fail.')
    elif vram >= 8:
        print('✅ 8+ GB VRAM — full feature set available including longer outputs.')
else:
    print("""
    ⚠️ WARNING: No GPU detected.

    OmniVoice requires a GPU. To change runtime:
        1. Runtime → Change runtime type
        2. Select T4 GPU
        3. Save and reconnect
    """)

## Step 2: Install OmniVoice

In [ ]:
# Install OmniVoice (Colab already has compatible PyTorch+CUDA)
!pip install -q omnivoice
print('✅ OmniVoice installed')

# Install cloudflared for public URL tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ Cloudflared ready')

## Step 3: Launch OmniVoice Demo

The model (`k2-fsa/OmniVoice`) downloads automatically on first run (~few GB).

The Gradio interface includes:
- **Zero-Shot Voice Cloning** — upload any audio sample (3 seconds is enough)
- **Cross-Lingual TTS** — clone a voice in one language, speak in another (600+ languages)
- **Voice Design** — build a voice from scratch: Gender, Age, Accent, Style
- **Batch Inference** — process multiple texts at once

> **Tip:** Leave language on **Auto** for best cross-lingual results.

In [ ]:
import subprocess, re, threading, time

demo = subprocess.Popen(
    'omnivoice-demo --ip 0.0.0.0 --port 8001',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
cf = subprocess.Popen(
    '/content/cloudflared tunnel --url http://localhost:8001 --no-autoupdate',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

url_pattern = re.compile(r'(https://[a-z0-9-]+\.trycloudflare\.com)')

def watch_demo():
    for line in demo.stdout:
        print('[omnivoice]', line.strip())

def watch_cf():
    for line in cf.stdout:
        m = url_pattern.search(line)
        if m:
            print(f'\n✅ OmniVoice Public URL: {m.group(1)}')
            print('Share this link — works on any device until the Colab session ends.\n')

threading.Thread(target=watch_demo, daemon=True).start()
threading.Thread(target=watch_cf, daemon=True).start()

print('Launching OmniVoice... wait ~30 seconds for the public URL to appear.')
while True:
    time.sleep(1)

## Tips & Feature Guide

### Voice Cloning (CLI alternative)
```bash
omnivoice-infer --model k2-fsa/OmniVoice \
    --text "Bienvenido a nuestra clínica dental" \
    --ref_audio ref.wav \
    --output output.wav
```

### Voice Design (CLI alternative)
```bash
omnivoice-infer --model k2-fsa/OmniVoice \
    --text "Hello, welcome to our service" \
    --instruct "female, warm, Latin American Spanish accent" \
    --output output.wav
```

### Cross-Lingual TTS (Perfect for Lima Clients)
- Upload an English voice sample → output Spanish speech in that same voice
- OmniVoice supports 600+ languages natively

### Batch Processing
```bash
omnivoice-infer-batch --model k2-fsa/OmniVoice --input batch.json --output-dir ./outputs
```